In [1]:
import numpy as np
import os #Tạo thư mục, Đọc file, Quản lý dataset
import tensorflow
from matplotlib import pyplot as plt
import mediapipe as mp #Trích xuất keypoints bàn tay, Phát hiện pose, face, hand landmarks
import time
import cv2 #Mở webcam, Xử lý hình ảnh, Hiển thị video realtime

C:\Users\PC\anaconda3\envs\gesture_ai\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
mp_holistic = mp.solutions.holistic # Gọi Mô hình Holistic của MediaPipe
mp_drawing = mp.solutions.drawing_utils #công cụ vẽ landmark lên ảnh

In [5]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) #Chuyển đổi hệ màu từ BGR -> RGB
    image.flags.writeable = False #Đặt ảnh ở chế độ không cho chỉnh sửa
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

In [7]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS)
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    

In [9]:
def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS,
                                mp_drawing.DrawingSpec(color=(0, 255, 255), thickness=1, circle_radius=1),
                                mp_drawing.DrawingSpec(color=(0, 200, 200), thickness=1, circle_radius=1))
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=3),
                                mp_drawing.DrawingSpec(color=(0, 180, 0), thickness=2, circle_radius=2))
    #Trái - Xanh dương 
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=3),
                                mp_drawing.DrawingSpec(color=(200, 0, 0), thickness=2, circle_radius=2))
    #Phải - Đỏ
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=3),
                                mp_drawing.DrawingSpec(color=(0, 0, 200), thickness=2, circle_radius=2))
    

In [11]:
#hàm trích xuất đặc trưng cho 1 frame.
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() \
                if results.pose_landmarks else np.zeros(33*4)

    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() \
                    if results.face_landmarks else np.zeros(468*3)
    #Tay phải
    hand_right = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() \
                    if results.right_hand_landmarks else np.zeros(21*3)
    #Tay trái
    hand_left = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() \
                    if results.left_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, hand_left, hand_right])

In [33]:
DATA_PATH = os.path.join('MP_DATA_20')
actions = np.array(['Thich','an'])
## 30 video dữ liệu
num_sequences = 100
#mỗi video dài 30 frame
sequence_length = 30

In [35]:
#Tạo cấu trúc thư mục để lưu dữ liệu train
for action in actions:
    for sequence in range(num_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

In [37]:
cap = cv2.VideoCapture(0) 
cap.set(3, 1280) 
cap.set(4, 720)

stop = False

# Khởi tạo mô hình MediaPipe Holistic
with mp_holistic.Holistic(model_complexity=1,min_detection_confidence=0.5,min_tracking_confidence=0.5) as holistic:

    for action in actions:
        for sequence in range(num_sequences): #Mỗi hành động có 30 video
            for frame_num in range(sequence_length): #Mỗi video có 30 frame

                # Đọc từng frame từ camera
                ret, frame = cap.read()
                if not ret:
                    stop = True
                    break
        
                # Gửi frame vào MediaPipe để nhận diện
                image, results = mediapipe_detection(frame, holistic)
                # print(results)
        
                # Vẽ landmarks lên ảnh
                draw_styled_landmarks(image, results)
                

                #HIỂN THỊ THÔNG BÁO TRÊN MÀN HÌNH
                if frame_num == 0:
                    #Viết chữ lên màn hình với tọa độ (120,200)
                    cv2.putText(image, 'BAT DAU THU THAP', (120,200),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 1, cv2.LINE_AA)
                    cv2.putText(image, 'Dang thu thap frame {} cho video so {}'.format(action, sequence), (15,12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 2, cv2.LINE_AA)
                    
                    cv2.imshow('My Camera', image)
                    
                    # Chờ 2 giây nhưng vẫn cho ESC hoạt động
                    start_time = cv2.getTickCount()
                    while (cv2.getTickCount() - start_time) / cv2.getTickFrequency() < 2:
                        if cv2.waitKey(1) & 0xFF == 27:
                            stop = True
                            break
                    if stop:
                        break
                else:
                    cv2.putText(image, 'Dang thu thap frame {} cho video so {}'.format(action, sequence), (15,12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 2, cv2.LINE_AA)
                    # Hiển thị camera
                    cv2.imshow('My Camera', image)

                #Lưu keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                # Kiểm tra ESC mỗi frame
                if cv2.waitKey(10) & 0xFF == 27:
                    stop = True
                    break

            if stop:
                break

        if stop:
            break

cap.release()
cv2.destroyAllWindows()